- Seven fit setups on the 135 LEGA-C DR2 quiescent refit targets.
- (a) October refit reused; (d) reuses 61 carbon-mask fits.
- (f), (g): 61 Mg b galaxies refitted below Mg b; other 74 reused.
- Fit seed 20260832 everywhere. Bootstrap seed 20261110, 10000 draws.
- Age: posterior median mass-weighted age from derived outputs.
- C/N quantiles: weighted nested-sampling samples of `ceridwen_result.h5`.
- Mg b group: October fit has fitted pixels in the Mg b bandpass.
- Mg b window: [Trager et al. 1998, Table 2](https://arxiv.org/pdf/astro-ph/9712258), bandpass plus pseudo-continua, air.
- (h): baseline on the (e) window, all 135.
- (i), (j): baseline and C/N free on the chosen wider window.
- (k), (l), (m): (i) window extended blue or red; window galaxies covering it.

In [1]:
from pathlib import Path
import json
import sys
import h5py
import jax
import numpy as np
import pandas as pd
import astropy.units as u
from astropy.cosmology import Planck18
from scipy.stats import linregress, spearmanr

jax.config.update('jax_enable_x64', True)
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts/experiment.py').exists())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'scripts'))
from scripts.experiment import validate_result
from absorption_mask_analysis import weighted_quantiles
from ceridwen.fit import load_result_h5
from ceridwen.cosmology import age_gyr, E_of_z, resolve_cosmology
from ceridwen.observation.absorption_features import feature_windows
from ceridwen.observation.filters import air2vac

OUT = ROOT / 'results/chronometer-135-2026-11-10'
REFIT = ROOT / 'results/dr2-refit-2026-10-12'
CARBON = ROOT / 'results/carbon-mask-2026-11-08'
TARGETS = [t for t in json.loads((REFIT / 'targets.json').read_text())['targets'] if 'folder' in t]
assert len(TARGETS) == 135
MGB_61 = set(json.loads((OUT / 'cn-free-noMgb/experiment.json').read_text())['targets'])
MGB_WINDOW_REST_VAC = np.asarray(air2vac(np.array([5142.625, 5206.375])))  # Trager+98 Table 2, Mg b


def run_dir(run, arm, t):
    return run / 'run/fits' / arm / f"{t['object_id']}-{t['spect_id']}"


def first(*paths):
    return next((p for p in paths if p.exists()), paths[-1])


ARMS = {
    'a_baseline': lambda t: REFIT / t['folder'],
    'b_cn_free': lambda t: run_dir(OUT / 'cn-free', 'cn_free', t),
    'c_cn_afe0p25': lambda t: run_dir(OUT / 'cn-afe0p25', 'cn_afe0p25', t),
    'd_carbon_mask': lambda t: first(run_dir(CARBON / 'check', 'carbon_mask', t), run_dir(CARBON / 'rest', 'carbon_mask', t),
                                     run_dir(OUT / 'carbon-mask', 'carbon_mask', t)),
    'e_cn_afe0p25_common': lambda t: run_dir(OUT / 'cn-afe0p25-common', 'cn_afe0p25_common', t),
    'f_cn_free_noMgb': lambda t: (run_dir(OUT / 'cn-free-noMgb', 'cn_free_noMgb', t) if t['spect_id'] in MGB_61
                                  else ARMS['b_cn_free'](t)),
    'g_baseline_noMgb': lambda t: (run_dir(OUT / 'baseline-noMgb', 'baseline_noMgb', t) if t['spect_id'] in MGB_61
                                   else ARMS['a_baseline'](t)),
}
FIXED_AFE = {'c_cn_afe0p25': 0.25, 'e_cn_afe0p25_common': 0.25}
ORIG = list(ARMS)
W_CONFIG = json.loads((OUT / 'baseline-w110/experiment.json').read_text())
W110 = set(W_CONFIG['targets'])
ARMS |= {
    'h_baseline_common': lambda t: run_dir(OUT / 'baseline-common', 'baseline_common', t),
    'i_baseline_w110': lambda t: run_dir(OUT / 'baseline-w110', 'baseline_w110', t),
    'j_cn_free_w110': lambda t: run_dir(OUT / 'cn-free-w110', 'cn_free_w110', t),
}
WINDOW_ARMS = ['e_cn_afe0p25_common', 'h_baseline_common', 'i_baseline_w110', 'j_cn_free_w110']
LADDER = {'k_baseline_blue': ('baseline-blue', 'baseline_blue'), 'l_baseline_red_hb': ('baseline-red-hb', 'baseline_red_hb'),
          'm_baseline_red_mgb': ('baseline-red-mgb', 'baseline_red_mgb')}
LADDER_CONFIG = {arm: json.loads((OUT / d / 'experiment.json').read_text()) for arm, (d, _) in LADDER.items()}
LADDER_RANGE = {arm: LADDER_CONFIG[arm]['arms'][name]['settings']['rest_range'] for arm, (_, name) in LADDER.items()}
for arm, (d, name) in LADDER.items():
    ARMS[arm] = lambda t, d=d, name=name: run_dir(OUT / d, name, t)
SUBSET = {'i_baseline_w110': W110, 'j_cn_free_w110': W110} | {arm: set(c['targets']) for arm, c in LADDER_CONFIG.items()}

- Validate every fit; failures stay in the table, out of statistics.
- `fit_rest_*`: first and last fitted pixel, rest vacuum Å.

In [2]:
def fit_row(arm, t):
    directory = ARMS[arm](t)
    row = dict(arm=arm, target=t['spect_id'], path=str(directory.relative_to(ROOT)))
    if not directory.exists():
        return row | dict(valid=False, failure='no result directory')
    try:
        validate_result(directory)
        row |= dict(valid=True, failure='')
    except Exception as error:
        return row | dict(valid=False, failure=str(error).split(':')[0])
    source = json.loads((directory.parents[2] / 'manifest.json').read_text())['source']
    row |= dict(astro_commit=source['commit'][:8], ceridwen_commit=source['submodules']['ceridwen'][:8])
    with h5py.File(directory / 'ceridwen_derived_outputs.h5') as f:
        names = f['summary/parameter'].asstr()[:]
        q = {k: dict(zip(names, f[f'summary/{k}'][:])) for k in ('q16', 'q50', 'q84')}
        row |= dict(z=float(f.attrs['redshift']), seed=int(f.attrs['random_seed']))
        for label, name in (('age_gyr', 'mass-weighted age [Gyr]'), ('feh', 'logzsol'),
                            ('afe', 'afe'), ('tau_dust', 'diffuse_tau_noll')):
            for k in q:
                row[f'{label}_{k}'] = float(q[k][name]) if name in q[k] else FIXED_AFE.get(arm, np.nan)
        wave = f['spectrum/wavelength'][:].astype(float)
        mask = f['spectrum/mask'][:].astype(bool)
    rest = wave / (1 + row['z'])
    lo, hi = feature_windows(['Mgb'], zred=row['z'])[0]
    row |= dict(fitted_pixels=int(mask.sum()), fit_rest_min=rest[mask].min(), fit_rest_max=rest[mask].max(),
                Mgb_n=int(np.sum(mask & (wave >= lo) & (wave <= hi))),
                Mgb_window_n=int(np.sum(mask & (rest >= MGB_WINDOW_REST_VAC[0]) & (rest <= MGB_WINDOW_REST_VAC[1]))))
    if arm in ('b_cn_free', 'c_cn_afe0p25', 'e_cn_afe0p25_common', 'f_cn_free_noMgb', 'j_cn_free_w110'):
        fit = load_result_h5(directory / 'ceridwen_result.h5')
        for p in ('cfe', 'nfe'):
            values = fit.samples.get(p)
            quantiles = weighted_quantiles(np.asarray(values), np.asarray(fit.log_weights)) if values is not None else [np.nan] * 3
            row |= dict(zip((f'{p}_q16', f'{p}_q50', f'{p}_q84'), quantiles))
    return row


fits = pd.DataFrame([fit_row(arm, t) for arm in ARMS for t in TARGETS if arm not in SUBSET or t['spect_id'] in SUBSET[arm]])
base = fits[fits.arm == 'a_baseline'].set_index('target')
fits['has_Mgb'] = fits.target.map(base.Mgb_n > 0)
fits['z'] = fits.target.map(base.z)
assert set(base.index[base.Mgb_n > 0]) == MGB_61
assert (fits.seed.dropna() == 20260832).all()
cut = fits[fits.arm.isin(['f_cn_free_noMgb', 'g_baseline_noMgb']) & fits.target.isin(MGB_61) & fits.valid]
assert (cut.Mgb_window_n == 0).all() and (cut.fit_rest_max < MGB_WINDOW_REST_VAC[0]).all()
window = fits[fits.arm.isin(WINDOW_ARMS) & fits.valid]
assert (window.Mgb_n == 0).all() and (window.Mgb_window_n == 0).all()
for arm, (lo_, hi_) in LADDER_RANGE.items():  # each extension arm fits only inside its window
    own = fits[(fits.arm == arm) & fits.valid]
    assert (own.fit_rest_min >= lo_ - 1).all() and (own.fit_rest_max <= hi_ + 1).all()
fits.to_csv(OUT / 'per-fit.csv', index=False)
validation = fits.groupby('arm').agg(fits=('valid', 'size'), valid=('valid', 'sum'))
validation['failed'] = validation.fits - validation.valid
validation['pins'] = fits.groupby('arm').apply(
    lambda g: '; '.join(f'{a}/{c}: {n}' for (a, c), n in g.groupby(['astro_commit', 'ceridwen_commit']).size().items()))
validation.to_csv(OUT / 'validation.csv')
validation

,fits,valid,failed,pins
arm,,,,
a_baseline,135,135,0,7033ef13/d7c0536f: 1; 85d3acbf/d7c0536f: 63; a...
b_cn_free,135,135,0,99c179ba/13791e0e: 135
c_cn_afe0p25,135,135,0,99c179ba/13791e0e: 135
d_carbon_mask,135,135,0,33a06784/a0b1f6ed: 61; 99c179ba/13791e0e: 74
e_cn_afe0p25_common,135,130,5,99c179ba/13791e0e: 130
f_cn_free_noMgb,135,135,0,99c179ba/13791e0e: 135
g_baseline_noMgb,135,135,0,85d3acbf/d7c0536f: 37; 99c179ba/13791e0e: 61; ...
h_baseline_common,135,130,5,99c179ba/13791e0e: 130
i_baseline_w110,110,110,0,99c179ba/13791e0e: 110


- (e): fitted pixels of each galaxy inside 4155.0–4438.1 Å.

In [3]:
common_range = fits[fits.arm == 'e_cn_afe0p25_common'][['target', 'z', 'valid', 'fitted_pixels', 'fit_rest_min', 'fit_rest_max']]
common_range = common_range.join(base[['fitted_pixels']].rename(columns={'fitted_pixels': 'baseline_fitted_pixels'}), on='target')
common_range.to_csv(OUT / 'common-range-pixels.csv', index=False)
common_range[['fitted_pixels', 'baseline_fitted_pixels']].describe().round(1)

,fitted_pixels,baseline_fitted_pixels
count,130.0,135.0
mean,658.3,3621.6
std,61.6,99.4
min,513.0,3272.0
25%,593.0,3561.0
50%,667.0,3604.0
75%,681.0,3635.5
max,785.0,3884.0


- Widest rest window fully inside the October fitted range of N galaxies.
- Edges rounded inward to 0.1 Å; features: whole catalogue window inside.
- Forecast slope σ: 0.8 Gyr / (rms z · √N).
- Choice: widest window with forecast σ ≤ 1.2 Gyr per unit z.

In [4]:
from ceridwen.observation.absorption_features import ABSORPTION_FEATURES
FEATURE_REST = feature_windows(None, zred=0.0)
lo, hi = base.fit_rest_min, base.fit_rest_max


def widest(need):
    edges = ((L, np.sort(hi[lo <= L].to_numpy())[::-1][need - 1]) for L in np.sort(lo.unique()) if (lo <= L).sum() >= need)
    L, H = max(edges, key=lambda w: w[1] - w[0])
    return np.ceil(L * 10) / 10, np.floor(H * 10) / 10


candidates = []
for need in (135, 129, 120, 110, 100, 90, 80):
    L, H = widest(need)
    cover = (lo <= L) & (hi >= H)
    z = base.z[cover]
    rms = z.std(ddof=0)
    candidates.append(dict(
        required=need, galaxies=int(cover.sum()), rest_lo=L, rest_hi=H, width=H - L, z_min=z.min(), z_max=z.max(), z_rms=rms,
        forecast_slope_sigma=0.8 / (rms * np.sqrt(cover.sum())), Mgb_group=int((base.Mgb_n[cover] > 0).sum()),
        features=' '.join(f.name for f, (l, u) in zip(ABSORPTION_FEATURES, FEATURE_REST) if l >= L and u <= H)))
candidates = pd.DataFrame(candidates)
chosen = candidates[candidates.forecast_slope_sigma <= 1.2].sort_values('width').iloc[-1]
assert np.allclose([chosen.rest_lo, chosen.rest_hi], W_CONFIG['arms']['baseline_w110']['settings']['rest_range'])
assert set(base.index[(lo <= chosen.rest_lo) & (hi >= chosen.rest_hi)]) == W110
assert np.allclose(candidates.set_index('required').loc[129, ['rest_lo', 'rest_hi']].to_numpy(float), [4155.0, 4438.1])
candidates.to_csv(OUT / 'common-windows.csv', index=False)
candidates.round(3)

,required,galaxies,rest_lo,rest_hi,width,z_min,z_max,z_rms,forecast_slope_sigma,Mgb_group,features
0,135,135,4155.0,4206.6,51.6,0.603,0.987,0.101,0.681,61,
1,129,129,4155.0,4438.1,283.1,0.603,0.987,0.096,0.737,61,HgA Ca4227 G4300 Fe4383
2,120,120,4131.3,4573.8,442.5,0.603,0.987,0.085,0.864,60,HgA Ca4227 Ca4455 CN4150 G4300 Fe4383 Fe4531
3,110,110,4074.5,4666.1,591.6,0.603,0.984,0.076,0.999,56,HdA HgA Ca4227 Ca4455 CN4150 G4300 Fe4383 Fe4531
4,100,100,4103.4,4892.4,789.0,0.603,0.894,0.066,1.213,59,HgA Hbeta Ca4227 Ca4455 CN4150 G4300 C4668 Fe4...
5,90,90,4033.4,4947.4,914.0,0.603,0.894,0.064,1.320,53,HdA HgA Hbeta Ca4227 Ca4455 CN4150 G4300 C4668...
6,80,80,4028.9,5014.7,985.8,0.603,0.894,0.062,1.451,52,HdA HgA Hbeta Ca4227 Ca4455 CN4150 G4300 C4668...


- Common sample: galaxies valid in arms (a)–(g).
- own_valid: each new arm on its own valid fits.
- w110: window galaxies valid in (a), (b), (e), (h), (i), (j).
- Same bootstrap draws for every arm within a group.
- Offset: Mg b minus no-Mg b median residual about one line.
- Step model: age = a + b z + c [Mg b], least squares.

In [5]:
ages = fits.pivot(index='target', columns='arm', values='age_gyr_q50').reindex(base.index)
valid_by_arm = fits.pivot(index='target', columns='arm', values='valid').reindex(base.index).fillna(False).astype(bool)
common = base.loc[valid_by_arm.index[valid_by_arm[ORIG].all(axis=1)].sort_values()]  # sorted as in the first run, for the same draws
GROUPS = {'all': common.index, 'with_Mgb': common.index[common.Mgb_n > 0],
          'without_Mgb': common.index[common.Mgb_n == 0]}
COSMO = resolve_cosmology(None)
dt_dz = jax.vmap(jax.grad(lambda z: age_gyr(z, COSMO)))
H_UNIT = (1 / u.Gyr).to_value(u.km / u.s / u.Mpc)


def hubble(slope, z):
    # Jimenez & Loeb 2002 eq. 2 with dt/dz = d(age)/dz: H = -1 / ((1 + z) dt/dz)
    return -H_UNIT / ((1 + z) * slope) if slope < 0 else np.nan


def line_fit(z, y, indices):
    fit = linregress(z, y)
    zc = z[indices] - z[indices].mean(axis=1, keepdims=True)
    with np.errstate(invalid='ignore', divide='ignore'):
        draws = np.sum(zc * y[indices], axis=1) / np.sum(zc**2, axis=1)
    q16, q50, q84 = np.nanquantile(draws, [0.16, 0.5, 0.84])
    return dict(slope=fit.slope, q16=q16, q50=q50, q84=q84,
                scatter=np.std(y - fit.intercept - fit.slope * z, ddof=2))


def offset_and_step(z, y, mgb):
    line = np.polyfit(z, y, 1)
    r = y - np.polyval(line, z)
    offset = np.median(r[mgb]) - np.median(r[~mgb])
    step = np.linalg.lstsq(np.column_stack([np.ones_like(z), z, mgb.astype(float)]), y, rcond=None)[0]
    return offset, step[1], step[2]


rng = np.random.default_rng(20261110)
INDICES = {g: rng.integers(0, len(ix), size=(10000, len(ix))) for g, ix in GROUPS.items()}


def group_index(ok):
    return {'all': base.index[ok], 'with_Mgb': base.index[ok & (base.Mgb_n > 0)],
            'without_Mgb': base.index[ok & (base.Mgb_n == 0)]}


def draws_for(ix):
    return np.random.default_rng(20261110).integers(0, len(ix), size=(10000, len(ix)))


def slope_row(sample, arm, group, index, indices, steps):
    z = base.loc[index, 'z'].to_numpy()
    y = ages.loc[index, arm].to_numpy()
    z_med = float(np.median(z))
    universe = line_fit(z, np.asarray(age_gyr(z, COSMO)), indices)
    s = line_fit(z, y, indices)
    sub = fits[(fits.arm == arm) & fits.target.isin(index)]
    rho = spearmanr(sub.z, sub.afe_q50) if sub.afe_q50.nunique() > 1 else (np.nan, np.nan)
    row = dict(
        sample=sample, group=group, arm=arm, galaxies=len(index), z_min=z.min(), z_max=z.max(), z_median=z_med,
        slope_gyr_per_z=s['slope'], bootstrap_q16=s['q16'], bootstrap_q50=s['q50'], bootstrap_q84=s['q84'],
        residual_scatter_gyr=s['scatter'],
        tuniv_line_slope_gyr_per_z=universe['slope'], tuniv_tangent_at_zmed_gyr_per_z=float(dt_dz(np.array([z_med]))[0]),
        H_kms_mpc=hubble(s['slope'], z_med), H_from_q16_slope=hubble(s['q16'], z_med),
        H_from_q84_slope=hubble(s['q84'], z_med), H_fit_cosmology_kms_mpc=float(COSMO.H0 * E_of_z(z_med, COSMO)),
        H_from_tuniv_line=hubble(universe['slope'], z_med),
        rho_afe_z=rho[0], rho_afe_z_p=rho[1],
        afe_median=sub.afe_q50.median(), age_median_gyr=sub.age_gyr_q50.median(),
        feh_median=sub.feh_q50.median(), cfe_median=sub.cfe_q50.median(), nfe_median=sub.nfe_q50.median())
    if steps:
        mgb = (base.loc[index, 'Mgb_n'] > 0).to_numpy()
        offset, step_slope, step = offset_and_step(z, y, mgb)
        draws = np.array([offset_and_step(z[i], y[i], mgb[i]) for i in indices])
        q = np.nanquantile(draws, [0.16, 0.84], axis=0)
        row |= dict(mgb_offset_gyr=offset, mgb_offset_q16=q[0, 0], mgb_offset_q84=q[1, 0],
                    step_model_slope_gyr_per_z=step_slope, step_model_slope_q16=q[0, 1], step_model_slope_q84=q[1, 1],
                    step_model_step_gyr=step, step_model_step_q16=q[0, 2], step_model_step_q84=q[1, 2],
                    H_step_model_kms_mpc=hubble(step_slope, z_med))
    return row


rows = [slope_row('common', arm, g, ix, INDICES[g], g == 'all') for g, ix in GROUPS.items() for arm in ORIG]
for arm in ORIG:  # each arm on its own valid fits, when that differs from the common sample
    if valid_by_arm[arm].all() and len(common) < len(base):
        for g, ix in group_index(valid_by_arm[arm]).items():
            rows.append(slope_row('all_135', arm, g, ix, draws_for(ix), g == 'all'))
for arm in ['h_baseline_common', 'i_baseline_w110', 'j_cn_free_w110']:
    for g, ix in group_index(valid_by_arm[arm]).items():
        rows.append(slope_row('own_valid', arm, g, ix, draws_for(ix), g == 'all'))
W_COMPARE = ['a_baseline', 'b_cn_free', 'e_cn_afe0p25_common', 'h_baseline_common', 'i_baseline_w110', 'j_cn_free_w110']
w_ok = valid_by_arm[W_COMPARE].all(axis=1) & base.index.isin(W110)
for g, ix in group_index(w_ok).items():
    for arm in W_COMPARE:
        rows.append(slope_row('w110', arm, g, ix, draws_for(ix), g == 'all'))
for arm in LADDER:  # own fits; then (a), (i) and the arm on the same galaxies
    for g, ix in group_index(valid_by_arm[arm]).items():
        if len(ix) >= 5:
            rows.append(slope_row('own_valid', arm, g, ix, draws_for(ix), False))
    same = valid_by_arm[['a_baseline', 'i_baseline_w110', arm]].all(axis=1)
    for g, ix in group_index(same).items():
        if len(ix) >= 5:
            for other in ['a_baseline', 'i_baseline_w110', arm]:
                rows.append(slope_row(f'{arm}_galaxies', other, g, ix, draws_for(ix), False))
slopes = pd.DataFrame(rows)
assert np.isclose(Planck18.H(slopes.z_median[0]).to_value('km/(s Mpc)'), slopes.H_fit_cosmology_kms_mpc[0], rtol=1e-3)
slopes.to_csv(OUT / 'age-redshift-slopes.csv', index=False)
slopes[['sample', 'group', 'arm', 'galaxies', 'slope_gyr_per_z', 'bootstrap_q16', 'bootstrap_q84', 'residual_scatter_gyr',
        'H_kms_mpc', 'rho_afe_z', 'afe_median', 'age_median_gyr', 'feh_median', 'mgb_offset_gyr',
        'step_model_slope_gyr_per_z']].round(3)

,sample,group,arm,galaxies,slope_gyr_per_z,bootstrap_q16,bootstrap_q84,residual_scatter_gyr,H_kms_mpc,rho_afe_z,afe_median,age_median_gyr,feh_median,mgb_offset_gyr,step_model_slope_gyr_per_z
0,common,all,a_baseline,130,-1.335,-2.026,-0.638,0.821,423.785,-0.488,-0.030,4.057,-0.285,0.278,-1.307
1,common,all,b_cn_free,130,1.362,0.606,2.157,0.992,NaN,-0.274,0.123,4.436,-0.418,-0.830,-1.839
2,common,all,c_cn_afe0p25,130,1.557,0.785,2.371,1.038,NaN,NaN,0.250,4.379,-0.507,-0.820,-1.646
3,common,all,d_carbon_mask,130,-2.443,-3.142,-1.734,0.824,231.663,-0.580,0.095,4.393,-0.370,0.102,-2.689
4,common,all,e_cn_afe0p25_common,130,-3.767,-4.418,-3.093,0.799,150.231,NaN,0.250,4.930,-0.434,-0.168,-4.464
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93,m_baseline_red_mgb_galaxies,all,i_baseline_w110,46,-9.383,-12.112,-6.504,0.918,61.952,-0.236,0.216,4.998,-0.458,NaN,NaN
94,m_baseline_red_mgb_galaxies,all,m_baseline_red_mgb,46,-7.233,-9.914,-4.433,0.841,80.369,-0.275,0.061,4.078,-0.260,NaN,NaN
95,m_baseline_red_mgb_galaxies,with_Mgb,a_baseline,46,-4.436,-7.086,-1.569,0.792,131.037,-0.182,0.097,4.231,-0.238,NaN,NaN
96,m_baseline_red_mgb_galaxies,with_Mgb,i_baseline_w110,46,-9.383,-12.112,-6.504,0.918,61.952,-0.236,0.216,4.998,-0.458,NaN,NaN


- Paired per-galaxy age differences on fits valid in both arms.

In [6]:
PAIRS = [('b_cn_free', 'a_baseline'), ('c_cn_afe0p25', 'a_baseline'), ('d_carbon_mask', 'a_baseline'),
         ('e_cn_afe0p25_common', 'a_baseline'), ('f_cn_free_noMgb', 'a_baseline'), ('g_baseline_noMgb', 'a_baseline'),
         ('c_cn_afe0p25', 'b_cn_free'), ('e_cn_afe0p25_common', 'c_cn_afe0p25'),
         ('f_cn_free_noMgb', 'b_cn_free'), ('h_baseline_common', 'a_baseline'), ('e_cn_afe0p25_common', 'h_baseline_common'),
         ('i_baseline_w110', 'a_baseline'), ('j_cn_free_w110', 'b_cn_free'), ('j_cn_free_w110', 'i_baseline_w110'),
         ('i_baseline_w110', 'h_baseline_common')]
paired = base[['z', 'Mgb_n']].copy()
rows = []
for new, old in PAIRS:
    ok = valid_by_arm[new] & valid_by_arm[old]
    paired[f'dage_{new}_minus_{old}'] = (ages[new] - ages[old]).where(ok)
    for group, index in {'all': base.index[ok], 'with_Mgb': base.index[ok & (base.Mgb_n > 0)],
                         'without_Mgb': base.index[ok & (base.Mgb_n == 0)]}.items():
        d = (ages.loc[index, new] - ages.loc[index, old]).to_numpy()
        z = base.loc[index, 'z'].to_numpy()
        s = line_fit(z, d, np.random.default_rng(20261110).integers(0, len(index), size=(10000, len(index))))
        rho = spearmanr(z, d) if np.ptp(d) > 0 else (np.nan, np.nan)
        rows.append(dict(pair=f'{new} - {old}', group=group, galaxies=len(d), dage_median=np.median(d),
                         dage_q16=np.quantile(d, 0.16), dage_q84=np.quantile(d, 0.84), older_fraction=np.mean(d > 0),
                         dage_vs_z_slope=s['slope'], dage_vs_z_q16=s['q16'], dage_vs_z_q84=s['q84'],
                         rho_dage_z=rho[0], rho_dage_z_p=rho[1]))
paired.to_csv(OUT / 'paired-age-differences-per-galaxy.csv')
paired_summary = pd.DataFrame(rows)
paired_summary.to_csv(OUT / 'paired-age-differences.csv', index=False)
paired_summary.round(3)

,pair,group,galaxies,dage_median,dage_q16,dage_q84,older_fraction,dage_vs_z_slope,dage_vs_z_q16,dage_vs_z_q84,rho_dage_z,rho_dage_z_p
0,b_cn_free - a_baseline,all,135,0.312,-0.537,1.002,0.704,2.526,2.007,3.061,0.363,0.000
1,b_cn_free - a_baseline,with_Mgb,61,-0.150,-0.920,0.483,0.443,0.007,-2.190,2.601,0.009,0.944
2,b_cn_free - a_baseline,without_Mgb,74,0.517,0.087,1.154,0.919,-0.712,-1.404,-0.040,-0.123,0.295
3,c_cn_afe0p25 - a_baseline,all,135,0.199,-0.612,0.904,0.637,2.731,2.208,3.263,0.399,0.000
4,c_cn_afe0p25 - a_baseline,with_Mgb,61,-0.177,-1.022,0.346,0.377,-0.636,-2.774,1.911,-0.055,0.673
5,c_cn_afe0p25 - a_baseline,without_Mgb,74,0.502,0.018,1.127,0.851,-0.365,-1.123,0.385,0.012,0.920
6,d_carbon_mask - a_baseline,all,135,0.250,-0.054,0.682,0.793,-1.171,-1.426,-0.907,-0.280,0.001
7,d_carbon_mask - a_baseline,with_Mgb,61,0.323,0.065,0.713,0.852,-1.990,-3.072,-1.020,-0.230,0.074
8,d_carbon_mask - a_baseline,without_Mgb,74,0.138,-0.072,0.598,0.743,-1.350,-1.713,-0.995,-0.365,0.001
9,e_cn_afe0p25_common - a_baseline,all,130,0.717,-0.164,1.509,0.792,-2.431,-3.105,-1.743,-0.247,0.005


- The 61 Mg b galaxies: Mg b cut minus full range, per galaxy.

In [7]:
QUANTITIES = ['afe_q50', 'age_gyr_q50', 'cfe_q50', 'nfe_q50']
wide = fits.set_index(['target', 'arm'])[QUANTITIES + ['z']].unstack('arm')
cut_rows, cut_summary = [], []
for new, old in [('f_cn_free_noMgb', 'b_cn_free'), ('g_baseline_noMgb', 'a_baseline')]:
    index = sorted(MGB_61 & set(base.index[valid_by_arm[new] & valid_by_arm[old]]))
    frame = pd.DataFrame({'target': index, 'pair': f'{new} - {old}', 'z': wide.loc[index, ('z', old)].to_numpy()})
    for q in QUANTITIES:
        frame[f'{q}_{old}'] = wide.loc[index, (q, old)].to_numpy()
        frame[f'{q}_{new}'] = wide.loc[index, (q, new)].to_numpy()
        frame[f'delta_{q}'] = frame[f'{q}_{new}'] - frame[f'{q}_{old}']
        d = frame[f'delta_{q}'].dropna()
        if len(d):
            cut_summary.append(dict(pair=f'{new} - {old}', quantity=q, galaxies=len(d), median=d.median(),
                                    q16=d.quantile(0.16), q84=d.quantile(0.84), positive_fraction=(d > 0).mean()))
    cut_rows.append(frame)
pd.concat(cut_rows).to_csv(OUT / 'mgb-cut-paired-per-galaxy.csv', index=False)
cut_summary = pd.DataFrame(cut_summary)
cut_summary.to_csv(OUT / 'mgb-cut-paired.csv', index=False)
cut_summary.round(3)

,pair,quantity,galaxies,median,q16,q84,positive_fraction
0,f_cn_free_noMgb - b_cn_free,afe_q50,61,-0.221,-0.392,-0.072,0.033
1,f_cn_free_noMgb - b_cn_free,age_gyr_q50,61,0.155,-0.216,0.929,0.705
2,f_cn_free_noMgb - b_cn_free,cfe_q50,61,-0.123,-0.288,-0.001,0.164
3,f_cn_free_noMgb - b_cn_free,nfe_q50,61,0.013,-0.090,0.223,0.639
4,g_baseline_noMgb - a_baseline,afe_q50,61,-0.106,-0.202,-0.015,0.049
5,g_baseline_noMgb - a_baseline,age_gyr_q50,61,-0.091,-0.500,0.134,0.344


- Age width: posterior 84th minus 16th percentile, per galaxy.
- Prior age: production SFH prior median at each redshift (cn-response experiment 9).

In [8]:
prior = pd.read_csv(ROOT / 'results/cn-response-2026-11-10/prior-slope/prior-per-galaxy.csv').set_index('target').loc[base.index]
assert np.allclose(prior.redshift, base.z, atol=1e-6)
assert prior.sort_values('redshift').prior_age_q50.is_monotonic_decreasing
width = fits.assign(width=fits.age_gyr_q84 - fits.age_gyr_q16).pivot(index='target', columns='arm', values='width').reindex(base.index)
prior_width = prior.prior_age_q84 - prior.prior_age_q16
rows = []
for arm in ['a_baseline'] + WINDOW_ARMS:
    for sample, ok in (('own_valid', valid_by_arm[arm]), ('w110', w_ok)):
        index = base.index[ok]
        ratio = width.loc[index, arm] / width.loc[index, 'a_baseline']
        rho = spearmanr(ages.loc[index, arm], prior.loc[index, 'prior_age_q50'])
        rows.append(dict(arm=arm, sample=sample, galaxies=len(index), width_ratio_median=ratio.median(),
                         width_ratio_q16=ratio.quantile(0.16), width_ratio_q84=ratio.quantile(0.84),
                         age_width_median_gyr=width.loc[index, arm].median(),
                         posterior_over_prior_width_median=(width.loc[index, arm] / prior_width[index]).median(),
                         prior_median_age_slope=linregress(base.z[index], prior.loc[index, 'prior_age_q50']).slope,
                         rho_age_prior_median=rho[0], rho_age_prior_median_p=rho[1]))
age_width = pd.DataFrame(rows)
age_width.to_csv(OUT / 'age-width-prior.csv', index=False)
age_width.round(3)

,arm,sample,galaxies,width_ratio_median,width_ratio_q16,width_ratio_q84,age_width_median_gyr,posterior_over_prior_width_median,prior_median_age_slope,rho_age_prior_median,rho_age_prior_median_p
0,a_baseline,own_valid,135,1.000,1.000,1.000,0.746,0.201,-2.279,0.165,0.055
1,a_baseline,w110,108,1.000,1.000,1.000,0.776,0.201,-2.339,0.152,0.116
2,e_cn_afe0p25_common,own_valid,130,1.619,1.203,2.178,1.165,0.308,-2.285,0.482,0.000
3,e_cn_afe0p25_common,w110,108,1.581,1.195,2.226,1.171,0.307,-2.339,0.411,0.000
4,h_baseline_common,own_valid,130,1.546,1.199,2.129,1.149,0.308,-2.285,0.445,0.000
5,h_baseline_common,w110,108,1.517,1.198,2.204,1.159,0.307,-2.339,0.353,0.000
6,i_baseline_w110,own_valid,110,1.290,0.982,1.779,0.966,0.259,-2.340,0.333,0.000
7,i_baseline_w110,w110,108,1.288,0.978,1.782,0.951,0.257,-2.339,0.333,0.000
8,j_cn_free_w110,own_valid,110,1.273,1.024,1.770,0.980,0.257,-2.340,0.387,0.000
9,j_cn_free_w110,w110,108,1.266,1.019,1.771,0.980,0.257,-2.339,0.393,0.000


- (k), (l), (m) minus (i), per galaxy, on fits valid in both.
- Band pull: sum of fitted pulls in band / √(fitted band pixels).
- Same metric as cn-response residual stack; median across galaxies.
- Added features: catalogue windows inside the arm's window, not inside (i)'s.

In [9]:
I_RANGE = W_CONFIG['arms']['baseline_w110']['settings']['rest_range']
H_RANGE = json.loads((OUT / 'baseline-common/experiment.json').read_text())['arms']['baseline_common']['settings']['rest_range']
EXT = {'afe': 'afe_q50', 'age': 'age_gyr_q50', 'feh': 'feh_q50'}
value = fits.pivot(index='target', columns='arm', values=list(EXT.values())).reindex(base.index)
rows, per_galaxy = [], base[['z', 'Mgb_n']].copy()
for arm in LADDER:
    ok = valid_by_arm[arm] & valid_by_arm['i_baseline_w110']
    for label, col in EXT.items():
        per_galaxy[f'd{label}_{arm}_minus_i'] = (value[(col, arm)] - value[(col, 'i_baseline_w110')]).where(ok)
    for group, index in group_index(ok).items():
        if len(index) < 5:
            continue
        z = base.loc[index, 'z'].to_numpy()
        row = dict(arm=arm, group=group, galaxies=len(index))
        for label in EXT:
            d = per_galaxy.loc[index, f'd{label}_{arm}_minus_i'].to_numpy()
            row |= {f'd{label}_median': np.median(d), f'd{label}_q16': np.quantile(d, 0.16), f'd{label}_q84': np.quantile(d, 0.84)}
        s = line_fit(z, per_galaxy.loc[index, f'dage_{arm}_minus_i'].to_numpy(), draws_for(index))
        rho = spearmanr(z, per_galaxy.loc[index, f'dage_{arm}_minus_i'])
        rows.append(row | dict(dage_vs_z_slope=s['slope'], dage_vs_z_q16=s['q16'], dage_vs_z_q84=s['q84'],
                               rho_dage_z=rho[0], rho_dage_z_p=rho[1]))
per_galaxy.dropna(how='all', subset=[c for c in per_galaxy if c.startswith('d')]).to_csv(OUT / 'window-extension-paired-per-galaxy.csv')
extension = pd.DataFrame(rows)
extension.to_csv(OUT / 'window-extension-paired.csv', index=False)
extension.round(3)

,arm,group,galaxies,dafe_median,dafe_q16,dafe_q84,dage_median,dage_q16,dage_q84,dfeh_median,dfeh_q16,dfeh_q84,dage_vs_z_slope,dage_vs_z_q16,dage_vs_z_q84,rho_dage_z,rho_dage_z_p
0,k_baseline_blue,all,60,-0.177,-0.324,-0.066,0.013,-0.375,0.389,0.016,-0.059,0.102,0.474,-0.267,1.269,0.065,0.624
1,k_baseline_blue,with_Mgb,7,-0.185,-0.237,-0.158,0.238,-0.229,0.513,-0.038,-0.061,0.038,19.005,12.992,24.596,0.750,0.052
2,k_baseline_blue,without_Mgb,53,-0.177,-0.337,-0.064,0.012,-0.376,0.343,0.017,-0.058,0.129,0.619,-0.233,1.524,0.075,0.594
3,l_baseline_red_hb,all,96,-0.118,-0.263,-0.037,0.006,-0.342,0.325,0.011,-0.071,0.090,0.280,-0.336,0.899,-0.008,0.940
4,l_baseline_red_hb,with_Mgb,56,-0.136,-0.251,-0.042,-0.026,-0.350,0.377,0.009,-0.067,0.089,-0.426,-1.552,0.728,-0.120,0.380
5,l_baseline_red_hb,without_Mgb,40,-0.107,-0.285,-0.038,0.059,-0.315,0.273,0.017,-0.071,0.093,0.552,-0.266,1.381,0.113,0.489
6,m_baseline_red_mgb,all,46,-0.163,-0.288,-0.001,-0.964,-1.369,-0.289,0.214,0.091,0.289,2.150,0.450,3.780,0.133,0.377
7,m_baseline_red_mgb,with_Mgb,46,-0.163,-0.288,-0.001,-0.964,-1.369,-0.289,0.214,0.091,0.289,2.150,0.450,3.780,0.133,0.377


In [10]:
def added_features(arm):
    lo_, hi_ = LADDER_RANGE[arm]
    return [f.name for f, (l, h) in zip(ABSORPTION_FEATURES, FEATURE_REST)
            if lo_ <= l and h <= hi_ and not (I_RANGE[0] <= l and h <= I_RANGE[1])]


def band_pulls(directory, names):
    with h5py.File(directory / 'ceridwen_derived_outputs.h5') as f:
        z = float(f.attrs['redshift'])
        wave = f['spectrum/wavelength'][:].astype(float)
        mask = f['spectrum/mask'][:].astype(bool)
        pull = f['spectrum/pull'][:]
    out = {}
    for name, (lo_, hi_) in zip(names, feature_windows(names, zred=z)):
        sel = mask & (wave >= lo_) & (wave <= hi_)
        out[name] = (pull[sel].sum() / np.sqrt(sel.sum()) if sel.any() else np.nan, int(sel.sum()))
    return out


BY_ID = {t['spect_id']: t for t in TARGETS}
rows = []
for arm in LADDER:
    names = added_features(arm)
    index = base.index[valid_by_arm[[arm, 'i_baseline_w110', 'a_baseline']].all(axis=1)]
    for fit_arm in (arm, 'a_baseline'):
        for target in index:
            for name, (metric, n) in band_pulls(ARMS[fit_arm](BY_ID[target]), names).items():
                rows.append(dict(arm=arm, fit=fit_arm, target=target, feature=name, band_pull=metric, fitted_pixels=n))
pulls = pd.DataFrame(rows)
pulls.to_csv(OUT / 'window-extension-band-pulls-per-galaxy.csv', index=False)
band = pulls.dropna(subset=['band_pull']).groupby(['arm', 'feature', 'fit'], sort=False).agg(
    galaxies=('band_pull', 'size'), median=('band_pull', 'median'), q16=('band_pull', lambda x: x.quantile(0.16)),
    q84=('band_pull', lambda x: x.quantile(0.84)), median_pixels=('fitted_pixels', 'median')).reset_index()
band = pulls[['arm', 'feature', 'fit']].drop_duplicates().merge(band, how='left').fillna({'galaxies': 0})
band.to_csv(OUT / 'window-extension-band-pulls.csv', index=False)
band.round(2)

,arm,feature,fit,galaxies,median,q16,q84,median_pixels
0,k_baseline_blue,H10,k_baseline_blue,60.0,-0.40,-1.68,0.43,73.0
1,k_baseline_blue,H9,k_baseline_blue,60.0,0.80,-0.30,2.02,74.0
2,k_baseline_blue,H8,k_baseline_blue,60.0,0.70,-0.57,1.88,75.0
3,k_baseline_blue,CaK,k_baseline_blue,0.0,NaN,NaN,NaN,NaN
4,k_baseline_blue,CaH,k_baseline_blue,0.0,NaN,NaN,NaN,NaN
5,k_baseline_blue,H10,a_baseline,60.0,-2.34,-3.54,-0.69,73.0
6,k_baseline_blue,H9,a_baseline,60.0,0.31,-0.94,1.38,74.0
7,k_baseline_blue,H8,a_baseline,60.0,1.47,0.25,3.09,75.0
8,k_baseline_blue,CaK,a_baseline,0.0,NaN,NaN,NaN,NaN
9,k_baseline_blue,CaH,a_baseline,0.0,NaN,NaN,NaN,NaN


In [11]:
WINDOWS = {'a_baseline': ('all_135', None), 'h_baseline_common': ('own_valid', H_RANGE), 'i_baseline_w110': ('own_valid', I_RANGE)}
WINDOWS |= {arm: ('own_valid', LADDER_RANGE[arm]) for arm in LADDER}
ladder = []
for arm, (sample, rest) in WINDOWS.items():
    r = slopes[(slopes['sample'] == sample) & (slopes.group == 'all') & (slopes.arm == arm)].iloc[0]
    ladder.append(dict(arm=arm, rest_lo=rest[0] if rest else np.nan, rest_hi=rest[1] if rest else np.nan,
                       galaxies=r.galaxies, afe_median=r.afe_median, slope_gyr_per_z=r.slope_gyr_per_z,
                       bootstrap_q16=r.bootstrap_q16, bootstrap_q84=r.bootstrap_q84,
                       tuniv_line_slope_gyr_per_z=r.tuniv_line_slope_gyr_per_z))
ladder = pd.DataFrame(ladder)
ladder.to_csv(OUT / 'window-ladder.csv', index=False)
ladder.round(3)

,arm,rest_lo,rest_hi,galaxies,afe_median,slope_gyr_per_z,bootstrap_q16,bootstrap_q84,tuniv_line_slope_gyr_per_z
0,a_baseline,NaN,NaN,135,-0.031,-1.551,-2.229,-0.878,-5.172
1,h_baseline_common,4155.0,4438.1,130,0.251,-3.260,-3.909,-2.617,-5.186
2,i_baseline_w110,4074.5,4666.1,110,0.211,-4.601,-5.559,-3.598,-5.309
3,k_baseline_blue,3742.9,4666.1,60,0.013,-4.116,-5.439,-2.732,-5.132
4,l_baseline_red_hb,4074.5,4893.0,96,0.101,-4.593,-5.934,-3.099,-5.460
5,m_baseline_red_mgb,4074.5,5207.9,46,0.061,-7.233,-9.914,-4.433,-5.919


- Pin check: same-pin baseline of the carbon-mask check against October.

In [12]:
check = []
for t in TARGETS:
    directory = run_dir(CARBON / 'check', 'baseline', t)
    if directory.exists():
        with h5py.File(directory / 'ceridwen_derived_outputs.h5') as f:
            q = dict(zip(f['summary/parameter'].asstr()[:], f['summary/q50'][:]))
        b = base.loc[t['spect_id']]
        check.append(dict(target=t['spect_id'], dage=q['mass-weighted age [Gyr]'] - b.age_gyr_q50,
                          dafe=q['afe'] - b.afe_q50, dfeh=q['logzsol'] - b.feh_q50))
pin_check = pd.DataFrame(check)
pin_check.to_csv(OUT / 'pin-check.csv', index=False)
pin_check.describe().loc[['count', 'mean', 'min', 'max']].round(4)

,dage,dafe,dfeh
count,10.0000,10.0000,10.0000
mean,0.0371,-0.0010,-0.0027
min,-0.0520,-0.0128,-0.0172
max,0.1487,0.0064,0.0036
